In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql import Row





StatementMeta(, 27ccd13d-e737-4a96-9c73-419440615549, 5, Finished, Available, Finished, False)

In [ ]:
customers = spark.table("Retail_Lakehouse.dbo.olist_customers_dataset")
display(customers.limit(10))

In [9]:
customers_clean = (
    customers
    .withColumnRenamed("customer_city", "city")
    .withColumnRenamed("customer_state", "state")
    .withColumnRenamed("customer_zip_code_prefix", "zip_code")
    .withColumn("zip_code", col("zip_code").cast("int"))
    .withColumn("city", initcap(col("city")))
    .dropDuplicates(["customer_id"])
    )

display(customers_clean.limit(10))

customers_clean.write.format("delta").mode("overwrite").saveAsTable("Retail_Silver.dbo.customers")

StatementMeta(, dc6ea024-8ef7-49e8-a4e4-ab41294f03b3, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c4d467c-b666-4ddf-9335-b8c76bae9b42)

In [3]:
items = spark.table("Retail_Lakehouse.dbo.olist_order_items_dataset")
display(items.limit(10))

StatementMeta(, 85f95f8a-6a43-4eb5-b98f-14ba0c42bebf, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3635fb42-39f7-4977-931c-e60287824d1d)

In [5]:
items_clean = (
    items
    .withColumnRenamed("order_item_id", "item_id")
    .withColumn("item_id", col("item_id").cast("int"))
    .withColumn("price", col("price").cast("float"))
    .withColumn("freight_value", col("freight_value").cast("float"))
    .withColumn("shipping_limit_date", to_date(col("shipping_limit_date")))
    .dropDuplicates(["order_id", "item_id"])
)

display(items_clean.limit(10))
items_clean.write.format("delta").mode("overwrite").saveAsTable("Retail_Silver.dbo.items")

StatementMeta(, 85f95f8a-6a43-4eb5-b98f-14ba0c42bebf, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aa9785a5-1d40-49de-9c9d-cb7ca0e1d4f2)

In [2]:
# from pyspark.sql.functions import col
# items = spark.table("Retail_Lakehouse.dbo.olist_order_items_dataset")
# # 1. Rename and cast the columns first
# items_prepped = (
#     items
#     .withColumnRenamed("order_item_id", "item_id")
#     .withColumn("item_id", col("item_id").cast("int"))
# )

# # 2. Find and count the duplicate combinations
# duplicates = (
#     items_prepped
#     .groupBy("order_id", "item_id")
#     .count()
#     .filter(col("count") > 1)
# )

# # 3. Display the duplicate rows and total count
# print(f"Total duplicate combinations found: {duplicates.count()}")
# display(duplicates.orderBy(col("count").desc()))


StatementMeta(, f654476f-8969-4f66-8467-832d5f876067, 4, Finished, Available, Finished, False)

Total duplicate combinations found: 0


SynapseWidget(Synapse.DataFrame, 67f168cc-ef0c-4517-88f0-f6c0dc03d07a)

In [ ]:
location = spark.table("olist_geolocation_dataset")
display(location.limit(10))

StatementMeta(, 51e76513-a2a3-49c5-a1bc-24a5afdde37a, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ae596fa-9aa6-46f2-a961-5bb1948ba5d4)

In [ ]:
location = location.drop("geolocation_lat", "geolocation_lng")
location_clean = (
    location
    .withColumnRenamed("geolocation_zip_code_prefix", "zip_code")
    .withColumnRenamed("geolocation_city", "city")
    .withColumn("city", initcap(col("city")))
    .withColumn("zip_code", col("zip_code").cast("int"))
    .dropDuplicates(["zip_code"])
    )

display(location_clean.limit(10))

location_clean.write.format("delta").mode("overwrite").saveAsTable("Retail_Silver.dbo.location")

StatementMeta(, 51e76513-a2a3-49c5-a1bc-24a5afdde37a, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, db2717f5-5ddf-4d4e-8388-e9b16cb0b58d)

In [1]:
orders = spark.table("olist_orders_dataset")
display(orders.limit(10))

StatementMeta(, 9ea446fb-1d2f-4db0-805e-551f4f6d4c98, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 57df89c2-5a6f-422f-8191-e1942a907fc6)

In [4]:
orders = orders.drop("order_approved_at", "order_delivered_carrier_date")
orders_clean = (
    orders
    .withColumnsRenamed({"order_purchase_timestamp":"purchase_date", "order_delivered_customer_date":"delivery_date", "order_estimated_delivery_date":"estimated_delivery_date"})
    .withColumn("purchase_date", to_date(col("purchase_date")))
    .withColumn("delivery_date", to_date(col("delivery_date")))
    .withColumn("estimated_delivery_date", to_date(col("estimated_delivery_date")))
    .dropDuplicates(["order_id"])

)

display(orders_clean.limit(10))

orders_clean.write.format("delta").mode("overwrite").saveAsTable("Retail_Silver.dbo.orders")

StatementMeta(, 9ea446fb-1d2f-4db0-805e-551f4f6d4c98, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 65daf39d-bbd0-4183-9007-ea44282871f3)

In [ ]:
payments = spark.table("olist_order_payments_dataset")

display(payments.limit(10))

StatementMeta(, 27ccd13d-e737-4a96-9c73-419440615549, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 89d0b748-23e8-406d-a34d-8920892d5908)

In [ ]:
payments_clean = (
    payments
    .withColumnsRenamed({"payment_installments":"installments", "payment_value":"value"})
    .withColumns({"payment_sequential":col("payment_sequential").cast("int"), "installments":col("installments").cast("int"), "value":col("value").cast("float")})
    #.dropDuplicates(["order_id"])
)

display(payments_clean.limit(10))

payments_clean.write.format("delta").mode("overwrite").saveAsTable("Retail_Silver.dbo.payments")

StatementMeta(, 27ccd13d-e737-4a96-9c73-419440615549, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bfb25ddb-9c95-4d40-9328-70107728f935)

In [9]:
reviews = spark.table("olist_order_reviews_dataset")

display(reviews.limit(10))

StatementMeta(, 9ea446fb-1d2f-4db0-805e-551f4f6d4c98, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 49348c97-a3a5-43ac-a0b3-aa2304309270)

In [10]:
reviews = reviews.drop("review_creation_date", "review_answer_timestamp")

reviews_clean = (
    reviews
    .withColumn("review_score", col("review_score").cast("int"))
    .dropDuplicates(["review_id"])
)

display(reviews_clean.limit(10))

reviews_clean.write.format("delta").mode("overwrite").saveAsTable("Retail_Silver.dbo.reviews")

StatementMeta(, 9ea446fb-1d2f-4db0-805e-551f4f6d4c98, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 961a8ba4-4321-472a-822b-17caede5d45a)

In [4]:
products = spark.table("olist_products_dataset")

display(products.limit(10))

StatementMeta(, 6daa4445-2c07-409b-b77f-c15e18f1ef04, 6, Finished, Available, Finished, False)

INFO:trident_token_library_wrapper:Calling getAccessToken from PyTridentTokenLibrary


SynapseWidget(Synapse.DataFrame, b8f61bdb-fbb7-47d5-a9e2-1a6c4c1ab85c)

In [6]:
products = products.drop("product_name_lenght", "product_description_lenght")
products_clean = (
    products
    .withColumnsRenamed({"product_category_name":"category", "product_photos_qty":"photo_qty", "product_weight_g":"weight", "product_length_cm":"length", "product_height_cm":"height", "product_width_cm":"width"})
    .withColumns({"photo_qty":col("photo_qty").cast("int"), "weight":col("weight").cast("float"), "length":col("length").cast("int"), "height":col("height").cast("int"), "width":col("width").cast("int")})
    .dropDuplicates(["product_id"])
    )

display(products_clean.limit(10))

products_clean.write.format("delta").mode("overwrite").saveAsTable("Retail_Silver.dbo.products")

StatementMeta(, 6daa4445-2c07-409b-b77f-c15e18f1ef04, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 41833839-e525-47ca-9ca2-73d78d30c1aa)

In [12]:
sellers = spark.table("olist_sellers_dataset")

display(sellers.limit(10))

StatementMeta(, 6daa4445-2c07-409b-b77f-c15e18f1ef04, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81184b78-268e-4089-926c-4c7170f6a490)

In [15]:
seller_clean = (
    sellers
    .withColumnRenamed("seller_zip_code_prefix", "seller_zip_code")
    .withColumn("seller_zip_code", col("seller_zip_code").cast("int"))
    .dropDuplicates(["seller_id"])
)

display(seller_clean.limit(10))

seller_clean.write.format("delta").mode("overwrite").saveAsTable("Retail_Silver.dbo.sellers")

StatementMeta(, 6daa4445-2c07-409b-b77f-c15e18f1ef04, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56a70dd8-fe6d-42c4-b76a-2a754ed66084)

In [16]:
product_category = spark.table("product_category_name_translation")

category_clean = (
    product_category
    .dropDuplicates(["product_category_name"])
)

category_clean.write.format("delta").mode("overwrite").saveAsTable("Retail_Silver.dbo.prod_category")

StatementMeta(, 6daa4445-2c07-409b-b77f-c15e18f1ef04, 18, Finished, Available, Finished, False)